In [2]:
import pandas as pd

economic = pd.read_csv("D:/Projects/volatility-radar/data/raw/economic_calendar_raw.csv", delimiter=",")
eurusd = pd.read_csv("D:/Projects/volatility-radar/data/raw/EURUSD_daily.csv", delimiter=",")
gbpusd = pd.read_csv("D:/Projects/volatility-radar/data/raw/GBPUSD_daily.csv", delimiter=",")
usdjpy = pd.read_csv("D:/Projects/volatility-radar/data/raw/USDJPY_daily.csv", delimiter=",")

economic.head()

,date,time,currency,event,impact,actual,forecast,previous
0,Mon Jan 4,6:00am,JPY,Final Manufacturing PMI,Low,50.0,49.7,49.7
1,Mon Jan 4,1:45pm,EUR,Spanish Manufacturing PMI,Low,51.0,52.6,49.8
2,Mon Jan 4,2:15pm,EUR,Italian Manufacturing PMI,Low,52.8,53.5,51.5
3,Mon Jan 4,2:20pm,EUR,French Final Manufacturing PMI,Low,51.1,51.1,51.1
4,Mon Jan 4,2:25pm,EUR,German Final Manufacturing PMI,Low,58.3,58.6,58.6


In [3]:
print(eurusd.head(3))
print(eurusd.dtypes)
print(eurusd.shape)

    timestamp    open    high     low   close
0  2026-03-13  1.1510  1.1529  1.1410  1.1416
1  2026-03-12  1.1567  1.1575  1.1508  1.1510
2  2026-03-11  1.1611  1.1645  1.1559  1.1566
timestamp        str
open         float64
high         float64
low          float64
close        float64
dtype: object
(1356, 5)


In [4]:
print(economic.isnull().sum())
print(eurusd.isnull().sum())
print(gbpusd.isnull().sum())
print(usdjpy.isnull().sum())

date           0
time        5240
currency       0
event          0
impact         0
actual        43
forecast    1337
previous       1
dtype: int64
timestamp    0
open         0
high         0
low          0
close        0
dtype: int64
timestamp    0
open         0
high         0
low          0
close        0
dtype: int64
timestamp    0
open         0
high         0
low          0
close        0
dtype: int64


In [5]:
print(economic.dtypes)
print(eurusd.dtypes)
print(gbpusd.dtypes)
print(usdjpy.dtypes)

date        str
time        str
currency    str
event       str
impact      str
actual      str
forecast    str
previous    str
dtype: object
timestamp        str
open         float64
high         float64
low          float64
close        float64
dtype: object
timestamp        str
open         float64
high         float64
low          float64
close        float64
dtype: object
timestamp        str
open         float64
high         float64
low          float64
close        float64
dtype: object


In [6]:
import pandas as pd
import calendar

# split "Mon Jan 4" into parts
parts = economic['date'].str.split(' ', expand=True)

# map month abbreviation to number
month_map = {month: index for index, month in enumerate(calendar.month_abbr) if month}
economic['month_num'] = parts[1].map(month_map)
economic['day_num'] = parts[2].astype(int)

# detect year boundaries
year_change = ((economic['month_num'] == 1) & (economic['month_num'].shift(1) == 12)).cumsum()
economic['year'] = 2021 + year_change

# build proper datetime
economic['date'] = pd.to_datetime(dict(year=economic['year'], month=economic['month_num'], day=economic['day_num']))

# drop helper columns
economic.drop(columns=['month_num', 'day_num', 'year'], inplace=True)

print(economic['date'].head(10))
print(economic['date'].dtype)

0   2021-01-04
1   2021-01-04
2   2021-01-04
3   2021-01-04
4   2021-01-04
5   2021-01-04
6   2021-01-04
7   2021-01-04
8   2021-01-04
9   2021-01-04
Name: date, dtype: datetime64[us]
datetime64[us]


In [7]:
print(economic['date'].head(10))
print(economic['date'].dtype)
print(economic.columns.tolist())

0   2021-01-04
1   2021-01-04
2   2021-01-04
3   2021-01-04
4   2021-01-04
5   2021-01-04
6   2021-01-04
7   2021-01-04
8   2021-01-04
9   2021-01-04
Name: date, dtype: datetime64[us]
datetime64[us]
['date', 'time', 'currency', 'event', 'impact', 'actual', 'forecast', 'previous']


In [8]:
print(economic['date'].dtype)
print(economic['date'].head(10))
print(economic['date'].tail(10))

datetime64[us]
0   2021-01-04
1   2021-01-04
2   2021-01-04
3   2021-01-04
4   2021-01-04
5   2021-01-04
6   2021-01-04
7   2021-01-04
8   2021-01-04
9   2021-01-04
Name: date, dtype: datetime64[us]
13752   2026-03-19
13753   2026-03-19
13754   2026-03-19
13755   2026-03-19
13756   2026-03-20
13757   2026-03-20
13758   2026-03-20
13759   2026-03-20
13760   2026-03-20
13761   2026-03-20
Name: date, dtype: datetime64[us]


In [9]:
economic = economic[economic['date'] <= '2026-03-13']
print(economic['date'].max())
print(economic.shape)

2026-03-13 00:00:00
(13721, 8)


In [10]:
economic.dropna(subset=["actual", "previous"], inplace= True)

In [11]:
null_mask = economic['time'].isnull()
above_mask = null_mask.shift(-1).fillna(False)
above_mask
result = economic[null_mask | above_mask]
result

,date,time,currency,event,impact,actual,forecast,previous
6,2021-01-04,3:00pm,GBP,Final Manufacturing PMI,Medium,57.5,57.3,57.3
7,2021-01-04,NaN,GBP,M4 Money Supply m/m,Low,0.8%,0.4%,0.7%
8,2021-01-04,NaN,GBP,Mortgage Approvals,Low,105K,82K,98K
9,2021-01-04,NaN,GBP,Net Lending to Individuals m/m,Low,4.1B,3.0B,3.8B
16,2021-01-05,2:30pm,EUR,M3 Money Supply y/y,Low,11.0%,10.6%,10.5%
...,...,...,...,...,...,...,...,...
13716,2026-03-13,NaN,USD,Personal Income m/m,Low,0.4%,0.5%,0.3%
13717,2026-03-13,NaN,USD,Personal Spending m/m,Low,0.4%,0.3%,0.4%
13718,2026-03-13,7:30pm,USD,JOLTS Job Openings,High,6.95M,6.76M,6.55M
13719,2026-03-13,NaN,USD,Prelim UoM Consumer Sentiment,Medium,55.5,55.0,56.6


In [12]:
economic['time'] = economic['time'].ffill()

In [15]:
economic.drop(columns=['forecast'], inplace= True)

In [16]:
economic.isnull().sum()

date        0
time        0
currency    0
event       0
impact      0
actual      0
previous    0
dtype: int64

In [17]:
import os
os.chdir("d:/Projects/volatility-radar")
economic.to_csv("data/processed/economic_calendar_clean.csv", index=False)

In [18]:
# load all three
eurusd = pd.read_csv("data/raw/EURUSD_daily.csv")
gbpusd = pd.read_csv("data/raw/GBPUSD_daily.csv")
usdjpy = pd.read_csv("data/raw/USDJPY_daily.csv")

# convert timestamp to datetime and sort
for df, name in [(eurusd, 'EURUSD'), (gbpusd, 'GBPUSD'), (usdjpy, 'USDJPY')]:
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df.sort_values('timestamp', inplace=True)
    df.reset_index(drop=True, inplace=True)
    df.to_csv(f"data/processed/{name}_daily_clean.csv", index=False)
    print(f"{name}: {df.shape}, {df['timestamp'].min()} to {df['timestamp'].max()}")

EURUSD: (1356, 5), 2021-01-01 00:00:00 to 2026-03-13 00:00:00
GBPUSD: (1356, 5), 2021-01-01 00:00:00 to 2026-03-13 00:00:00
USDJPY: (1356, 5), 2021-01-01 00:00:00 to 2026-03-13 00:00:00
